In [1]:
import os
import duckdb
os.chdir('..')
import pandas as pd
from datetime import datetime, timezone

con = duckdb.connect('data/transit_kpi.duckdb')

In [2]:
# Verify tables are there
con.sql("SHOW TABLES").df()

,name
0,calendar
1,gtfsrt
2,stop_times
3,trips


In [3]:
con.sql("""
    SELECT
        service_id,
        monday,
        tuesday,
        wednesday,
        thursday,
        friday,
        saturday,
        sunday,
        start_date,
        end_date
    FROM calendar
""").df()

,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,910,1,1,1,1,1,0,0,20260329,20260403
1,810,0,0,0,0,0,0,0,20260329,20260509
2,812,0,0,0,0,0,0,0,20260329,20260509
3,813,0,0,0,0,0,0,0,20260329,20260509
4,940,1,1,1,1,1,0,0,20260329,20260509
5,942,0,0,0,0,0,1,0,20260329,20260509
6,943,0,0,0,0,0,0,1,20260329,20260509
7,46,1,1,1,1,1,0,0,20260329,20260606
8,48,0,0,0,0,0,1,0,20260329,20260606
9,49,0,0,0,0,0,0,1,20260329,20260606


In [3]:
from datetime import date

# Get Unix timestamp for midnight UTC on April 7th 2026
# This is the base we add scheduled seconds to
midnight = datetime(2026, 4, 7, 0, 0, 0, tzinfo=timezone.utc)
midnight_unix = int(midnight.timestamp())

print(f"Midnight April 7 UTC as Unix timestamp: {midnight_unix}")

Midnight April 7 UTC as Unix timestamp: 1775520000


In [4]:
con.sql("""
    SELECT service_id
    FROM calendar
    WHERE tuesday = 1
    AND start_date <= 20260407
    AND end_date >= 20260407
""").df()

,service_id
0,940
1,46
2,10
3,16
4,22
5,28
6,40
7,52


In [5]:
con.execute("DROP TABLE IF EXISTS calendar_dates")
con.execute("""
    CREATE TABLE calendar_dates AS
    SELECT *
    FROM read_csv_auto('data/google_bus/calendar_dates.txt')
""")

In [6]:
con.sql("""
    SELECT *
    FROM calendar_dates
    WHERE date = 20260407
""").df()

,service_id,date,exception_type
0,308052,20260407,1
1,308746,20260407,1
2,336010,20260407,1
3,33610,20260407,1
4,34010,20260407,1
5,340540,20260407,1
6,341046,20260407,1
7,341252,20260407,1
8,341610,20260407,1
9,390240,20260407,1


In [7]:
con.sql("""
    WITH active_services AS (
        -- Regular services running on Tuesday April 7th
        SELECT service_id
        FROM calendar
        WHERE tuesday = 1
        AND start_date <= 20260407
        AND end_date >= 20260407
        
        UNION
        
        -- Additional services added specifically on April 7th
        SELECT service_id
        FROM calendar_dates
        WHERE date = 20260407
        AND exception_type = 1
    ),
    matched AS (
        SELECT
            rt.trip_id,
            rt.route_id,
            rt.service_date,
            rt.snapshot_ts,
            rt.stop_sequence,
            rt.stop_id,
            st.arrival_time                 AS scheduled_arrival_time,
            st.arrival_seconds              AS scheduled_arrival_seconds,
            rt.predicted_unix,
            rt.predicted_ts,
            ROUND(
                (rt.predicted_unix - (1775520000 + st.arrival_seconds)) / 60.0
            , 1)                            AS delay_minutes
        FROM gtfsrt rt
        JOIN trips t
            ON rt.trip_id = t.trip_id
        JOIN active_services a
            ON t.service_id = a.service_id
        JOIN stop_times st
            ON rt.trip_id        = st.trip_id
            AND rt.stop_sequence = st.stop_sequence
        WHERE rt.route_id IN ('23', '47')
        AND rt.schedule_relationship = 0
    )
    SELECT *
    FROM matched
    WHERE delay_minutes BETWEEN -30 AND 60
    LIMIT 20
""").df()

,trip_id,route_id,service_date,snapshot_ts,stop_sequence,stop_id,scheduled_arrival_time,scheduled_arrival_seconds,predicted_unix,predicted_ts,delay_minutes


In [8]:
con.sql("""
    SELECT DISTINCT
        t.service_id,
        t.route_id
    FROM trips t
    WHERE t.route_id IN ('23', '47')
    ORDER BY t.route_id, t.service_id
""").df()

,service_id,route_id
0,10,23
1,12,23
2,13,23
3,10,47
4,12,47
5,13,47


In [10]:
print(con.sql("SELECT * FROM calendar WHERE service_id IN ('10', '12', '13')").df().to_string())

   service_id  monday  tuesday  wednesday  thursday  friday  saturday  sunday  start_date  end_date
0          10       1        1          1         1       1         0       0    20260329  20260613
1          12       0        0          0         0       0         1       0    20260329  20260613
2          13       0        0          0         0       0         0       1    20260329  20260613


In [11]:
print(con.sql("""
    SELECT COUNT(*) as matched_trips
    FROM gtfsrt rt
    JOIN trips t ON rt.trip_id = t.trip_id
    WHERE rt.route_id IN ('23', '47')
""").df().to_string())

   matched_trips
0         274892


In [12]:
print(con.sql("""
    SELECT DISTINCT
        t.service_id,
        COUNT(*) as records
    FROM gtfsrt rt
    JOIN trips t ON rt.trip_id = t.trip_id
    WHERE rt.route_id IN ('23', '47')
    GROUP BY t.service_id
    ORDER BY records DESC
""").df().to_string())

   service_id  records
0          10   274892


In [13]:
print(con.sql("""
    SELECT COUNT(*) as matched
    FROM gtfsrt rt
    JOIN trips t
        ON rt.trip_id = t.trip_id
    JOIN stop_times st
        ON rt.trip_id = st.trip_id
        AND rt.stop_sequence = st.stop_sequence
    WHERE rt.route_id IN ('23', '47')
    AND t.service_id = '10'
""").df().to_string())

   matched
0   274892


In [14]:
print(con.sql("""
    SELECT
        MIN(ROUND((rt.predicted_unix - (1775520000 + st.arrival_seconds)) / 60.0, 1)) as min_delay,
        MAX(ROUND((rt.predicted_unix - (1775520000 + st.arrival_seconds)) / 60.0, 1)) as max_delay,
        AVG(ROUND((rt.predicted_unix - (1775520000 + st.arrival_seconds)) / 60.0, 1)) as avg_delay,
        COUNT(*) as total
    FROM gtfsrt rt
    JOIN trips t
        ON rt.trip_id = t.trip_id
    JOIN stop_times st
        ON rt.trip_id        = st.trip_id
        AND rt.stop_sequence = st.stop_sequence
    WHERE rt.route_id IN ('23', '47')
    AND t.service_id = '10'
    AND rt.schedule_relationship = 0
""").df().to_string())

   min_delay  max_delay  avg_delay   total
0    -1212.9     1707.3  61.817162  274892


In [15]:
print(con.sql("""
    SELECT
        CASE
            WHEN delay < -30  THEN 'very early (< -30 min)'
            WHEN delay < -1   THEN 'early (-30 to -1 min)'
            WHEN delay <= 5   THEN 'on time (-1 to +5 min)'
            WHEN delay <= 15  THEN 'late (5 to 15 min)'
            WHEN delay <= 60  THEN 'very late (15 to 60 min)'
            ELSE                   'likely bad match (> 60 min)'
        END                  AS bucket,
        COUNT(*)             AS records,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM (
        SELECT
            ROUND(
                (rt.predicted_unix - (1775520000 + st.arrival_seconds)) / 60.0
            , 1) AS delay
        FROM gtfsrt rt
        JOIN trips t
            ON rt.trip_id = t.trip_id
        JOIN stop_times st
            ON rt.trip_id        = st.trip_id
            AND rt.stop_sequence = st.stop_sequence
        WHERE rt.route_id IN ('23', '47')
        AND t.service_id = '10'
        AND rt.schedule_relationship = 0
    )
    GROUP BY bucket
    ORDER BY MIN(delay)
""").df().to_string())

                        bucket  records   pct
0       very early (< -30 min)    35884  13.1
1  likely bad match (> 60 min)   239008  86.9


In [16]:
print(con.sql("""
    SELECT
        rt.trip_id,
        rt.snapshot_ts,
        rt.predicted_unix,
        rt.predicted_ts,
        st.arrival_time,
        st.arrival_seconds,
        1775520000 + st.arrival_seconds AS scheduled_unix,
        ROUND((rt.predicted_unix - (1775520000 + st.arrival_seconds)) / 60.0, 1) AS delay_minutes
    FROM gtfsrt rt
    JOIN stop_times st
        ON rt.trip_id = st.trip_id
        AND rt.stop_sequence = st.stop_sequence
    WHERE rt.route_id = '23'
    AND rt.schedule_relationship = 0
    AND rt.predicted_ts BETWEEN '2026-04-07T08:00:00+00:00' AND '2026-04-07T09:00:00+00:00'
    LIMIT 10
""").df().to_string())

  trip_id                snapshot_ts  predicted_unix               predicted_ts arrival_time  arrival_seconds  scheduled_unix  delay_minutes
0  662090  2026-04-07T08:35:25+00:00      1775550839  2026-04-07T08:33:59+00:00     04:23:00            15780      1775535780          251.0
1  662090  2026-04-07T08:35:25+00:00      1775550849  2026-04-07T08:34:09+00:00     04:23:17            15797      1775535797          250.9
2  662090  2026-04-07T08:35:25+00:00      1775550866  2026-04-07T08:34:26+00:00     04:23:37            15817      1775535817          250.8
3  662090  2026-04-07T08:35:25+00:00      1775550879  2026-04-07T08:34:39+00:00     04:23:59            15839      1775535839          250.7
4  662090  2026-04-07T08:35:25+00:00      1775550892  2026-04-07T08:34:52+00:00     04:24:18            15858      1775535858          250.6
5  662090  2026-04-07T08:35:25+00:00      1775550918  2026-04-07T08:35:18+00:00     04:24:38            15878      1775535878          250.7
6  662090  20

In [17]:
print(con.sql("""
    SELECT
        rt.trip_id,
        rt.snapshot_ts,
        rt.predicted_ts,
        st.arrival_time,
        st.arrival_seconds,
        -- Convert snapshot to seconds since midnight for comparison
        (EPOCH(CAST(rt.snapshot_ts AS TIMESTAMPTZ)) % 86400) AS snapshot_seconds,
        -- Difference between snapshot time and scheduled time in minutes
        ROUND(
            ABS(
                (EPOCH(CAST(rt.snapshot_ts AS TIMESTAMPTZ)) % 86400)
                - st.arrival_seconds
            ) / 60.0
        , 1) AS snapshot_vs_scheduled_minutes
    FROM gtfsrt rt
    JOIN stop_times st
        ON rt.trip_id        = st.trip_id
        AND rt.stop_sequence = st.stop_sequence
    WHERE rt.route_id = '23'
    AND rt.schedule_relationship = 0
    AND rt.predicted_ts BETWEEN '2026-04-07T08:00:00+00:00' AND '2026-04-07T09:00:00+00:00'
    LIMIT 10
""").df().to_string())

  trip_id                snapshot_ts               predicted_ts arrival_time  arrival_seconds  snapshot_seconds  snapshot_vs_scheduled_minutes
0  662090  2026-04-07T08:35:25+00:00  2026-04-07T08:33:59+00:00     04:23:00            15780           30925.0                          252.4
1  662090  2026-04-07T08:35:25+00:00  2026-04-07T08:34:09+00:00     04:23:17            15797           30925.0                          252.1
2  662090  2026-04-07T08:35:25+00:00  2026-04-07T08:34:26+00:00     04:23:37            15817           30925.0                          251.8
3  662090  2026-04-07T08:35:25+00:00  2026-04-07T08:34:39+00:00     04:23:59            15839           30925.0                          251.4
4  662090  2026-04-07T08:35:25+00:00  2026-04-07T08:34:52+00:00     04:24:18            15858           30925.0                          251.1
5  662090  2026-04-07T08:35:25+00:00  2026-04-07T08:35:18+00:00     04:24:38            15878           30925.0                          250.8

In [18]:
print(con.sql("""
    SELECT
        trip_id,
        stop_sequence,
        arrival_time,
        arrival_seconds
    FROM stop_times
    WHERE trip_id = '662090'
    ORDER BY stop_sequence
    LIMIT 10
""").df().to_string())

   trip_id  stop_sequence arrival_time  arrival_seconds
0   662090              1     04:23:00            15780
1   662090              2     04:23:17            15797
2   662090              3     04:23:37            15817
3   662090              4     04:23:59            15839
4   662090              5     04:24:18            15858
5   662090              6     04:24:38            15878
6   662090              7     04:24:59            15899
7   662090              8     04:25:17            15917
8   662090              9     04:25:43            15943
9   662090             10     04:25:58            15958


In [19]:
print(con.sql("""
    SELECT
        STRFTIME(CAST(predicted_ts AS TIMESTAMPTZ), '%H') AS predicted_hour,
        COUNT(*) as records
    FROM gtfsrt
    WHERE trip_id = '662090'
    GROUP BY predicted_hour
    ORDER BY predicted_hour
""").df().to_string())

  predicted_hour  records
0             04      372
1             05      399


In [20]:
print(con.sql("""
    SELECT DISTINCT
        rt.trip_id,
        rt.predicted_ts,
        st.arrival_time AS scheduled_first_stop
    FROM gtfsrt rt
    JOIN stop_times st
        ON rt.trip_id = st.trip_id
        AND st.stop_sequence = 1
    WHERE rt.route_id = '23'
    AND rt.predicted_ts BETWEEN '2026-04-07T08:00:00+00:00' 
                             AND '2026-04-07T09:00:00+00:00'
    ORDER BY rt.predicted_ts
    LIMIT 20
""").df().to_string())

   trip_id               predicted_ts scheduled_first_stop
0   662097  2026-04-07T08:00:01+00:00             27:32:00
1   662097  2026-04-07T08:00:06+00:00             27:32:00
2   662097  2026-04-07T08:00:11+00:00             27:32:00
3   662097  2026-04-07T08:00:17+00:00             27:32:00
4   662097  2026-04-07T08:00:41+00:00             27:32:00
5   662097  2026-04-07T08:00:46+00:00             27:32:00
6   662097  2026-04-07T08:00:56+00:00             27:32:00
7   662097  2026-04-07T08:01:08+00:00             27:32:00
8   662097  2026-04-07T08:01:43+00:00             27:32:00
9   662097  2026-04-07T08:02:22+00:00             27:32:00
10  662157  2026-04-07T08:02:42+00:00             04:00:00
11  662157  2026-04-07T08:03:00+00:00             04:00:00
12  662157  2026-04-07T08:03:28+00:00             04:00:00
13  662157  2026-04-07T08:03:40+00:00             04:00:00
14  662157  2026-04-07T08:04:13+00:00             04:00:00
15  662157  2026-04-07T08:04:19+00:00             04:00:

In [21]:
print(con.sql("""
    SELECT
        rt.trip_id,
        rt.predicted_ts,
        -- Convert UTC to Eastern (UTC-4 in April)
        CAST(
            EPOCH(CAST(rt.predicted_ts AS TIMESTAMPTZ)) - 14400
            AS INTEGER
        ) / 3600 % 24                           AS predicted_hour_eastern,
        st.arrival_time                         AS scheduled_first_stop,
        st.arrival_seconds / 3600.0             AS scheduled_hour
    FROM gtfsrt rt
    JOIN stop_times st
        ON rt.trip_id = st.trip_id
        AND st.stop_sequence = 1
    WHERE rt.route_id = '23'
    AND rt.predicted_ts BETWEEN '2026-04-07T08:00:00+00:00'
                             AND '2026-04-07T09:00:00+00:00'
    GROUP BY rt.trip_id, rt.predicted_ts, st.arrival_time, st.arrival_seconds
    ORDER BY rt.predicted_ts
    LIMIT 10
""").df().to_string())

  trip_id               predicted_ts  predicted_hour_eastern scheduled_first_stop  scheduled_hour
0  662097  2026-04-07T08:00:01+00:00                4.000278             27:32:00       27.533333
1  662097  2026-04-07T08:00:06+00:00                4.001667             27:32:00       27.533333
2  662097  2026-04-07T08:00:11+00:00                4.003056             27:32:00       27.533333
3  662097  2026-04-07T08:00:17+00:00                4.004722             27:32:00       27.533333
4  662097  2026-04-07T08:00:41+00:00                4.011389             27:32:00       27.533333
5  662097  2026-04-07T08:00:46+00:00                4.012778             27:32:00       27.533333
6  662097  2026-04-07T08:00:56+00:00                4.015556             27:32:00       27.533333
7  662097  2026-04-07T08:01:08+00:00                4.018889             27:32:00       27.533333
8  662097  2026-04-07T08:01:43+00:00                4.028611             27:32:00       27.533333
9  662097  2026-04-0

In [22]:
print(con.sql("""
    WITH base AS (
        SELECT
            rt.trip_id,
            rt.route_id,
            rt.service_date,
            rt.snapshot_ts,
            rt.stop_sequence,
            rt.stop_id,
            rt.predicted_unix,
            rt.predicted_ts,
            st.arrival_time,
            st.arrival_seconds,
            -- Compute midnight Unix dynamically from service_date
            EPOCH(CAST(rt.service_date AS DATE)) AS midnight_unix,
            -- Delay in minutes
            ROUND(
                (rt.predicted_unix - (EPOCH(CAST(rt.service_date AS DATE)) + st.arrival_seconds)) / 60.0
            , 1) AS delay_minutes
        FROM gtfsrt rt
        JOIN trips t
            ON rt.trip_id = t.trip_id
        JOIN stop_times st
            ON rt.trip_id        = st.trip_id
            AND rt.stop_sequence = st.stop_sequence
        WHERE rt.route_id IN ('23', '47')
        AND rt.schedule_relationship = 0
        AND t.service_id = '10'
    )
    SELECT
        CASE
            WHEN delay_minutes < -30  THEN 'very early (< -30 min)'
            WHEN delay_minutes < -1   THEN 'early (-30 to -1 min)'
            WHEN delay_minutes <= 5   THEN 'on time (-1 to +5 min)'
            WHEN delay_minutes <= 15  THEN 'late (5 to 15 min)'
            WHEN delay_minutes <= 60  THEN 'very late (15 to 60 min)'
            ELSE                           'likely bad match (> 60 min)'
        END                          AS bucket,
        COUNT(*)                     AS records,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM base
    GROUP BY bucket
    ORDER BY MIN(delay_minutes)
""").df().to_string())

                        bucket  records   pct
0       very early (< -30 min)    35884  13.1
1  likely bad match (> 60 min)   239008  86.9


In [23]:
print(con.sql("""
    SELECT
        service_date,
        EPOCH(CAST(service_date AS DATE)) AS computed_midnight,
        1775520000 AS hardcoded_midnight
    FROM gtfsrt
    WHERE route_id = '23'
    LIMIT 3
""").df().to_string())

  service_date  computed_midnight  hardcoded_midnight
0   2026-04-07       1.775520e+09          1775520000
1   2026-04-07       1.775520e+09          1775520000
2   2026-04-07       1.775520e+09          1775520000


In [24]:
print(con.sql("""
    SELECT DISTINCT
        rt.trip_id,
        rt.predicted_ts,
        st.arrival_time AS scheduled_first_stop
    FROM gtfsrt rt
    JOIN stop_times st
        ON rt.trip_id = st.trip_id
        AND st.stop_sequence = 1
    WHERE rt.route_id = '23'
    AND rt.predicted_ts BETWEEN '2026-04-07T13:00:00+00:00'
                             AND '2026-04-07T14:00:00+00:00'
    ORDER BY rt.predicted_ts
    LIMIT 10
""").df().to_string())

  trip_id               predicted_ts scheduled_first_stop
0  662227  2026-04-07T13:00:00+00:00             08:06:00
1  662118  2026-04-07T13:00:01+00:00             08:06:00
2  662122  2026-04-07T13:00:02+00:00             08:57:00
3  662229  2026-04-07T13:00:03+00:00             08:32:00
4  662091  2026-04-07T13:00:03+00:00             07:56:00
5  662118  2026-04-07T13:00:04+00:00             08:06:00
6  662087  2026-04-07T13:00:04+00:00             08:46:00
7  662091  2026-04-07T13:00:05+00:00             07:56:00
8  662120  2026-04-07T13:00:05+00:00             08:26:00
9  662116  2026-04-07T13:00:05+00:00             07:46:00


In [25]:
print(con.sql("""
    SELECT
        rt.trip_id,
        rt.predicted_ts,
        st.arrival_time,
        st.arrival_seconds,
        EPOCH(CAST(rt.service_date AS DATE)) AS midnight_unix,
        EPOCH(CAST(rt.service_date AS DATE)) + st.arrival_seconds AS scheduled_unix,
        rt.predicted_unix,
        ROUND(
            (rt.predicted_unix - (EPOCH(CAST(rt.service_date AS DATE)) + st.arrival_seconds)) / 60.0
        , 1) AS delay_minutes
    FROM gtfsrt rt
    JOIN stop_times st
        ON rt.trip_id        = st.trip_id
        AND rt.stop_sequence = st.stop_sequence
    WHERE rt.route_id = '23'
    AND rt.schedule_relationship = 0
    AND rt.predicted_ts BETWEEN '2026-04-07T13:00:00+00:00'
                             AND '2026-04-07T14:00:00+00:00'
    LIMIT 20
""").df().to_string())

   trip_id               predicted_ts arrival_time  arrival_seconds  midnight_unix  scheduled_unix  predicted_unix  delay_minutes
0   662087  2026-04-07T13:00:04+00:00     08:47:57            31677   1.775520e+09    1.775552e+09      1775566804          252.1
1   662087  2026-04-07T13:00:33+00:00     08:48:27            31707   1.775520e+09    1.775552e+09      1775566833          252.1
2   662087  2026-04-07T13:01:03+00:00     08:48:59            31739   1.775520e+09    1.775552e+09      1775566863          252.1
3   662087  2026-04-07T13:01:15+00:00     08:49:26            31766   1.775520e+09    1.775552e+09      1775566875          251.8
4   662087  2026-04-07T13:01:43+00:00     08:50:05            31805   1.775520e+09    1.775552e+09      1775566903          251.6
5   662087  2026-04-07T13:01:51+00:00     08:50:28            31828   1.775520e+09    1.775552e+09      1775566911          251.4
6   662087  2026-04-07T13:02:08+00:00     08:51:01            31861   1.775520e+09    1.77

In [26]:
print(con.sql("""
    SELECT
        rt.trip_id,
        rt.predicted_ts,
        st.arrival_time                                              AS scheduled_arrival,
        -- Convert scheduled_unix back to readable time
        STRFTIME(
            TO_TIMESTAMP(
                EPOCH(CAST(rt.service_date AS DATE)) + st.arrival_seconds
            ), '%Y-%m-%d %H:%M:%S'
        )                                                            AS scheduled_ts,
        rt.predicted_unix - (EPOCH(CAST(rt.service_date AS DATE)) + st.arrival_seconds) AS diff_seconds
    FROM gtfsrt rt
    JOIN stop_times st
        ON rt.trip_id        = st.trip_id
        AND rt.stop_sequence = st.stop_sequence
    WHERE rt.trip_id = '662087'
    AND rt.predicted_ts BETWEEN '2026-04-07T13:00:00+00:00'
                             AND '2026-04-07T13:01:00+00:00'
    LIMIT 3
""").df().to_string())

  trip_id               predicted_ts scheduled_arrival         scheduled_ts  diff_seconds
0  662087  2026-04-07T13:00:04+00:00          08:47:57  2026-04-07 04:47:57       15127.0
1  662087  2026-04-07T13:00:33+00:00          08:48:27  2026-04-07 04:48:27       15126.0
2  662087  2026-04-07T13:00:33+00:00          08:52:22  2026-04-07 04:52:22       14891.0


In [27]:
print(con.sql("""
    SELECT
        rt.trip_id,
        rt.predicted_ts,
        st.arrival_time                                              AS scheduled_arrival,
        STRFTIME(
            TO_TIMESTAMP(
                EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds
            ), '%Y-%m-%d %H:%M:%S'
        )                                                            AS scheduled_ts_fixed,
        ROUND(
            (rt.predicted_unix - (EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds)) / 60.0
        , 1)                                                         AS delay_minutes
    FROM gtfsrt rt
    JOIN stop_times st
        ON rt.trip_id        = st.trip_id
        AND rt.stop_sequence = st.stop_sequence
    WHERE rt.trip_id = '662087'
    AND rt.predicted_ts BETWEEN '2026-04-07T13:00:00+00:00'
                             AND '2026-04-07T13:01:00+00:00'
    LIMIT 5
""").df().to_string())

  trip_id               predicted_ts scheduled_arrival   scheduled_ts_fixed  delay_minutes
0  662087  2026-04-07T13:00:04+00:00          08:47:57  2026-04-07 08:47:57           12.1
1  662087  2026-04-07T13:00:33+00:00          08:48:27  2026-04-07 08:48:27           12.1
2  662087  2026-04-07T13:00:21+00:00          08:52:22  2026-04-07 08:52:22            8.0
3  662087  2026-04-07T13:00:32+00:00          08:52:52  2026-04-07 08:52:52            7.7
4  662087  2026-04-07T13:00:48+00:00          08:53:23  2026-04-07 08:53:23            7.4


In [28]:
print(con.sql("""
    WITH base AS (
        SELECT
            ROUND(
                (rt.predicted_unix - (EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds)) / 60.0
            , 1) AS delay_minutes
        FROM gtfsrt rt
        JOIN trips t
            ON rt.trip_id = t.trip_id
        JOIN stop_times st
            ON rt.trip_id        = st.trip_id
            AND rt.stop_sequence = st.stop_sequence
        WHERE rt.route_id IN ('23', '47')
        AND rt.schedule_relationship = 0
        AND t.service_id = '10'
    )
    SELECT
        CASE
            WHEN delay_minutes < -30 THEN 'very early (< -30 min)'
            WHEN delay_minutes < -1  THEN 'early (-30 to -1 min)'
            WHEN delay_minutes <= 5  THEN 'on time (-1 to +5 min)'
            WHEN delay_minutes <= 15 THEN 'late (5 to 15 min)'
            WHEN delay_minutes <= 60 THEN 'very late (15 to 60 min)'
            ELSE                          'likely bad match (> 60 min)'
        END          AS bucket,
        COUNT(*)     AS records,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM base
    GROUP BY bucket
    ORDER BY MIN(delay_minutes)
""").df().to_string())

                     bucket  records   pct
0    very early (< -30 min)    35884  13.1
1     early (-30 to -1 min)    53535  19.5
2    on time (-1 to +5 min)   125799  45.8
3        late (5 to 15 min)    45234  16.5
4  very late (15 to 60 min)    14440   5.3


In [29]:
print(con.sql("""
    SELECT
        rt.trip_id,
        rt.predicted_ts,
        st.arrival_time,
        st.arrival_seconds,
        ROUND(
            (rt.predicted_unix - (EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds)) / 60.0
        , 1) AS delay_minutes
    FROM gtfsrt rt
    JOIN trips t
        ON rt.trip_id = t.trip_id
    JOIN stop_times st
        ON rt.trip_id        = st.trip_id
        AND rt.stop_sequence = st.stop_sequence
    WHERE rt.route_id IN ('23', '47')
    AND rt.schedule_relationship = 0
    AND t.service_id = '10'
    AND ROUND(
            (rt.predicted_unix - (EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds)) / 60.0
        , 1) < -30
    ORDER BY delay_minutes
    LIMIT 20
""").df().to_string())

   trip_id               predicted_ts arrival_time  arrival_seconds  delay_minutes
0   662040  2026-04-07T06:43:22+00:00     26:56:14            96974        -1452.9
1   662040  2026-04-07T06:43:22+00:00     26:56:14            96974        -1452.9
2   662040  2026-04-07T06:43:22+00:00     26:56:14            96974        -1452.9
3   662040  2026-04-07T06:43:09+00:00     26:55:56            96956        -1452.8
4   662040  2026-04-07T06:46:10+00:00     26:59:00            97140        -1452.8
5   662040  2026-04-07T06:43:09+00:00     26:55:56            96956        -1452.8
6   662040  2026-04-07T06:43:09+00:00     26:55:56            96956        -1452.8
7   662040  2026-04-07T06:44:03+00:00     26:56:37            96997        -1452.6
8   662040  2026-04-07T06:43:00+00:00     26:55:38            96938        -1452.6
9   662040  2026-04-07T06:43:00+00:00     26:55:38            96938        -1452.6
10  662040  2026-04-07T06:47:52+00:00     27:00:29            97229        -1452.6
11  

In [30]:
print(con.sql("""
    SELECT
        rt.trip_id,
        rt.predicted_ts,
        st.arrival_time,
        st.arrival_seconds,
        -- For overnight trips (>24hrs), add an extra day to midnight
        CASE
            WHEN st.arrival_seconds >= 86400
            THEN EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds - 86400
            ELSE EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds
        END                                                          AS scheduled_unix_fixed,
        ROUND(
            (rt.predicted_unix - 
                CASE
                    WHEN st.arrival_seconds >= 86400
                    THEN EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds - 86400
                    ELSE EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds
                END
            ) / 60.0
        , 1)                                                         AS delay_minutes
    FROM gtfsrt rt
    JOIN trips t
        ON rt.trip_id = t.trip_id
    JOIN stop_times st
        ON rt.trip_id        = st.trip_id
        AND rt.stop_sequence = st.stop_sequence
    WHERE rt.trip_id = '662040'
    AND rt.predicted_ts BETWEEN '2026-04-07T06:43:00+00:00'
                             AND '2026-04-07T06:44:00+00:00'
    LIMIT 5
""").df().to_string())

  trip_id               predicted_ts arrival_time  arrival_seconds  scheduled_unix_fixed  delay_minutes
0  662040  2026-04-07T06:43:00+00:00     26:55:38            96938          1.775545e+09          -12.6
1  662040  2026-04-07T06:43:09+00:00     26:55:56            96956          1.775545e+09          -12.8
2  662040  2026-04-07T06:43:22+00:00     26:56:14            96974          1.775545e+09          -12.9
3  662040  2026-04-07T06:43:00+00:00     26:55:38            96938          1.775545e+09          -12.6
4  662040  2026-04-07T06:43:09+00:00     26:55:56            96956          1.775545e+09          -12.8


In [31]:
print(con.sql("""
    WITH base AS (
        SELECT
            ROUND(
                (rt.predicted_unix -
                    CASE
                        WHEN st.arrival_seconds >= 86400
                        THEN EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds - 86400
                        ELSE EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds
                    END
                ) / 60.0
            , 1) AS delay_minutes
        FROM gtfsrt rt
        JOIN trips t
            ON rt.trip_id = t.trip_id
        JOIN stop_times st
            ON rt.trip_id        = st.trip_id
            AND rt.stop_sequence = st.stop_sequence
        WHERE rt.route_id IN ('23', '47')
        AND rt.schedule_relationship = 0
        AND t.service_id = '10'
    )
    SELECT
        CASE
            WHEN delay_minutes < -30 THEN 'very early (< -30 min)'
            WHEN delay_minutes < -1  THEN 'early (-30 to -1 min)'
            WHEN delay_minutes <= 5  THEN 'on time (-1 to +5 min)'
            WHEN delay_minutes <= 15 THEN 'late (5 to 15 min)'
            WHEN delay_minutes <= 60 THEN 'very late (15 to 60 min)'
            ELSE                          'likely bad match (> 60 min)'
        END          AS bucket,
        COUNT(*)     AS records,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM base
    GROUP BY bucket
    ORDER BY MIN(delay_minutes)
""").df().to_string())

                     bucket  records   pct
0    very early (< -30 min)    27164   9.9
1     early (-30 to -1 min)    54727  19.9
2    on time (-1 to +5 min)   131681  47.9
3        late (5 to 15 min)    46853  17.0
4  very late (15 to 60 min)    14467   5.3


In [32]:
print(con.sql("""
    WITH base AS (
        SELECT
            ROUND(
                (rt.predicted_unix -
                    CASE
                        WHEN st.arrival_seconds >= 86400
                        THEN EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds - 86400
                        ELSE EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds
                    END
                ) / 60.0
            , 1) AS delay_minutes
        FROM gtfsrt rt
        JOIN trips t
            ON rt.trip_id = t.trip_id
        JOIN stop_times st
            ON rt.trip_id        = st.trip_id
            AND rt.stop_sequence = st.stop_sequence
        WHERE rt.route_id IN ('23', '47')
        AND rt.schedule_relationship = 0
        AND t.service_id = '10'
    )
    SELECT
        CASE
            WHEN delay_minutes < -60  THEN 'bad match (< -60 min)'
            WHEN delay_minutes < -30  THEN 'very early (-60 to -30 min)'
            WHEN delay_minutes < -1   THEN 'early (-30 to -1 min)'
            WHEN delay_minutes <= 5   THEN 'on time (-1 to +5 min)'
            WHEN delay_minutes <= 15  THEN 'late (5 to 15 min)'
            WHEN delay_minutes <= 60  THEN 'very late (15 to 60 min)'
            ELSE                           'bad match (> 60 min)'
        END          AS bucket,
        COUNT(*)     AS records,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM base
    GROUP BY bucket
    ORDER BY MIN(delay_minutes)
""").df().to_string())

                     bucket  records   pct
0     bad match (< -60 min)    27164   9.9
1     early (-30 to -1 min)    54727  19.9
2    on time (-1 to +5 min)   131681  47.9
3        late (5 to 15 min)    46853  17.0
4  very late (15 to 60 min)    14467   5.3


In [ ]:
con.execute("DROP TABLE IF EXISTS trip_delays")

con.execute("""
    CREATE TABLE trip_delays AS
    SELECT
        rt.trip_id,
        rt.route_id,
        rt.service_date,
        rt.snapshot_ts,
        rt.stop_sequence,
        rt.stop_id,
        rt.schedule_relationship,
        st.arrival_time                                              AS scheduled_arrival_time,
        st.arrival_seconds                                           AS scheduled_arrival_seconds,
        rt.predicted_unix,
        rt.predicted_ts,
        CASE
            WHEN st.arrival_seconds >= 86400
            THEN EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds - 86400
            ELSE EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds
        END                                                          AS scheduled_unix,
        ROUND(
            (rt.predicted_unix -
                CASE
                    WHEN st.arrival_seconds >= 86400
                    THEN EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds - 86400
                    ELSE EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds
                END
            ) / 60.0
        , 1)                                                         AS delay_minutes
    FROM gtfsrt rt
    JOIN trips t
        ON rt.trip_id = t.trip_id
    JOIN stop_times st
        ON rt.trip_id        = st.trip_id
        AND rt.stop_sequence = st.stop_sequence
    WHERE rt.route_id IN ('23', '47')
    AND rt.schedule_relationship = 0
    AND t.service_id = '10'
    -- Exclude confirmed bad matches
    AND ROUND(
            (rt.predicted_unix -
                CASE
                    WHEN st.arrival_seconds >= 86400
                    THEN EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds - 86400
                    ELSE EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds
                END
            ) / 60.0
        , 1) >= -60
""")

con.sql("SELECT COUNT(*) as rows FROM trip_delays").df()

,rows
0,247728


In [34]:
print(con.sql("""
    SELECT
        route_id,
        COUNT(*)                                                    AS total_records,
        SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5 THEN 1 ELSE 0 END) AS on_time_count,
        ROUND(
            SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)
        , 1)                                                        AS otp_pct
    FROM trip_delays
    GROUP BY route_id
    ORDER BY route_id
""").df().to_string())

  route_id  total_records  on_time_count  otp_pct
0       23         125268        72638.0     58.0
1       47         122460        59043.0     48.2


In [35]:
print(con.sql("""
    SELECT
        route_id,
        -- Strict: 0 early to 3 min late
        ROUND(
            SUM(CASE WHEN delay_minutes BETWEEN 0 AND 3 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)
        , 1) AS otp_0_3,
        -- Standard: 1 min early to 5 min late
        ROUND(
            SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)
        , 1) AS otp_1_5,
        -- Lenient: 2 min early to 7 min late
        ROUND(
            SUM(CASE WHEN delay_minutes BETWEEN -2 AND 7 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)
        , 1) AS otp_2_7
    FROM trip_delays
    GROUP BY route_id
    ORDER BY route_id
""").df().to_string())

  route_id  otp_0_3  otp_1_5  otp_2_7
0       23     29.2     58.0     75.7
1       47     23.9     48.2     65.4


In [36]:
print(con.sql("""
    SELECT
        rt.trip_id,
        rt.predicted_ts,
        st.arrival_time,
        st.arrival_seconds,
        ROUND(
            (rt.predicted_unix -
                CASE
                    WHEN st.arrival_seconds >= 86400
                    THEN EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds - 86400
                    ELSE EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds
                END
            ) / 60.0
        , 1) AS delay_minutes,
        -- How far is predicted time from scheduled time in hours
        ROUND(
            ABS(rt.predicted_unix -
                CASE
                    WHEN st.arrival_seconds >= 86400
                    THEN EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds - 86400
                    ELSE EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds
                END
            ) / 3600.0
        , 1) AS gap_hours
    FROM gtfsrt rt
    JOIN trips t
        ON rt.trip_id = t.trip_id
    JOIN stop_times st
        ON rt.trip_id        = st.trip_id
        AND rt.stop_sequence = st.stop_sequence
    WHERE rt.route_id IN ('23', '47')
    AND rt.schedule_relationship = 0
    AND t.service_id = '10'
    AND ROUND(
            (rt.predicted_unix -
                CASE
                    WHEN st.arrival_seconds >= 86400
                    THEN EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds - 86400
                    ELSE EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds
                END
            ) / 60.0
        , 1) < -60
    GROUP BY rt.trip_id, rt.predicted_ts, st.arrival_time, 
             st.arrival_seconds, rt.predicted_unix, rt.service_date
    ORDER BY delay_minutes
    LIMIT 20
""").df().to_string())

   trip_id               predicted_ts arrival_time  arrival_seconds  delay_minutes  gap_hours
0   728164  2026-04-07T01:11:34+00:00     21:18:11            76691        -1446.6       24.1
1   728164  2026-04-07T01:11:07+00:00     21:17:39            76659        -1446.5       24.1
2   728164  2026-04-07T01:09:35+00:00     21:16:03            76563        -1446.5       24.1
3   728164  2026-04-07T01:10:35+00:00     21:17:07            76627        -1446.5       24.1
4   728164  2026-04-07T01:09:06+00:00     21:15:32            76532        -1446.4       24.1
5   728164  2026-04-07T01:12:32+00:00     21:18:53            76733        -1446.4       24.1
6   728164  2026-04-07T01:13:31+00:00     21:19:55            76795        -1446.4       24.1
7   728164  2026-04-07T01:10:11+00:00     21:16:35            76595        -1446.4       24.1
8   728164  2026-04-07T01:08:49+00:00     21:15:00            76500        -1446.2       24.1
9   728164  2026-04-07T01:08:18+00:00     21:14:23          

In [37]:
print(con.sql("""
    WITH delay_calc AS (
        SELECT
            rt.trip_id,
            rt.predicted_ts,
            st.arrival_time,
            st.arrival_seconds,
            rt.predicted_unix,
            rt.service_date,
            -- Compute scheduled unix three ways:
            -- previous day, same day, next day
            EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds - 86400 AS scheduled_prev,
            EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds          AS scheduled_same,
            EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds + 86400 AS scheduled_next
        FROM gtfsrt rt
        JOIN trips t
            ON rt.trip_id = t.trip_id
        JOIN stop_times st
            ON rt.trip_id        = st.trip_id
            AND rt.stop_sequence = st.stop_sequence
        WHERE rt.route_id IN ('23', '47')
        AND rt.schedule_relationship = 0
        AND t.service_id = '10'
    ),
    best_match AS (
        SELECT
            *,
            -- Pick whichever scheduled unix is closest to predicted unix
            CASE
                WHEN ABS(predicted_unix - scheduled_prev) < ABS(predicted_unix - scheduled_same)
                 AND ABS(predicted_unix - scheduled_prev) < ABS(predicted_unix - scheduled_next)
                THEN scheduled_prev
                WHEN ABS(predicted_unix - scheduled_next) < ABS(predicted_unix - scheduled_same)
                THEN scheduled_next
                ELSE scheduled_same
            END AS best_scheduled_unix
        FROM delay_calc
    )
    SELECT
        CASE
            WHEN ROUND((predicted_unix - best_scheduled_unix) / 60.0, 1) < -60 THEN 'bad match (< -60 min)'
            WHEN ROUND((predicted_unix - best_scheduled_unix) / 60.0, 1) < -30 THEN 'very early (-60 to -30 min)'
            WHEN ROUND((predicted_unix - best_scheduled_unix) / 60.0, 1) < -1  THEN 'early (-30 to -1 min)'
            WHEN ROUND((predicted_unix - best_scheduled_unix) / 60.0, 1) <= 5  THEN 'on time (-1 to +5 min)'
            WHEN ROUND((predicted_unix - best_scheduled_unix) / 60.0, 1) <= 15 THEN 'late (5 to 15 min)'
            WHEN ROUND((predicted_unix - best_scheduled_unix) / 60.0, 1) <= 60 THEN 'very late (15 to 60 min)'
            ELSE                                                                     'bad match (> 60 min)'
        END          AS bucket,
        COUNT(*)     AS records,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM best_match
    GROUP BY bucket
    ORDER BY MIN(ROUND((predicted_unix - best_scheduled_unix) / 60.0, 1))
""").df().to_string())

                     bucket  records   pct
0     early (-30 to -1 min)    57607  21.0
1    on time (-1 to +5 min)   148269  53.9
2        late (5 to 15 min)    54326  19.8
3  very late (15 to 60 min)    14690   5.3


In [38]:
print(con.sql("""
    WITH delay_calc AS (
        SELECT
            rt.trip_id,
            rt.route_id,
            rt.service_date,
            rt.snapshot_ts,
            rt.stop_sequence,
            rt.stop_id,
            rt.schedule_relationship,
            rt.predicted_unix,
            rt.predicted_ts,
            st.arrival_time                                          AS scheduled_arrival_time,
            st.arrival_seconds                                       AS scheduled_arrival_seconds,
            -- Nearest day scheduled unix
            EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds - 86400 AS scheduled_prev,
            EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds          AS scheduled_same,
            EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds + 86400  AS scheduled_next
        FROM gtfsrt rt
        JOIN trips t
            ON rt.trip_id = t.trip_id
        JOIN stop_times st
            ON rt.trip_id        = st.trip_id
            AND rt.stop_sequence = st.stop_sequence
        WHERE rt.route_id IN ('23', '47')
        AND rt.schedule_relationship = 0
        AND t.service_id = '10'
    ),
    best_match AS (
        SELECT
            *,
            CASE
                WHEN ABS(predicted_unix - scheduled_prev) < ABS(predicted_unix - scheduled_same)
                 AND ABS(predicted_unix - scheduled_prev) < ABS(predicted_unix - scheduled_next)
                THEN scheduled_prev
                WHEN ABS(predicted_unix - scheduled_next) < ABS(predicted_unix - scheduled_same)
                THEN scheduled_next
                ELSE scheduled_same
            END AS scheduled_unix
        FROM delay_calc
    ),
    ranked AS (
        SELECT
            *,
            ROUND((predicted_unix - scheduled_unix) / 60.0, 1) AS delay_minutes,
            -- Rank snapshots per trip per stop by proximity to scheduled time
            ROW_NUMBER() OVER (
                PARTITION BY trip_id, stop_sequence
                ORDER BY ABS(predicted_unix - scheduled_unix)
            ) AS snapshot_rank
        FROM best_match
    )
    SELECT COUNT(*) as total, 
           COUNT(DISTINCT trip_id || '-' || stop_sequence) as unique_stop_events
    FROM ranked
    WHERE snapshot_rank = 1
""").df().to_string())

   total  unique_stop_events
0  42041               42041


In [39]:
con.execute("DROP TABLE IF EXISTS trip_delays")

con.execute("""
    CREATE TABLE trip_delays AS
    WITH delay_calc AS (
        SELECT
            rt.trip_id,
            rt.route_id,
            rt.service_date,
            rt.snapshot_ts,
            rt.stop_sequence,
            rt.stop_id,
            rt.schedule_relationship,
            rt.predicted_unix,
            rt.predicted_ts,
            st.arrival_time                                          AS scheduled_arrival_time,
            st.arrival_seconds                                       AS scheduled_arrival_seconds,
            EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds - 86400 AS scheduled_prev,
            EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds          AS scheduled_same,
            EPOCH(CAST(rt.service_date AS DATE)) + 14400 + st.arrival_seconds + 86400  AS scheduled_next
        FROM gtfsrt rt
        JOIN trips t
            ON rt.trip_id = t.trip_id
        JOIN stop_times st
            ON rt.trip_id        = st.trip_id
            AND rt.stop_sequence = st.stop_sequence
        WHERE rt.route_id IN ('23', '47')
        AND rt.schedule_relationship = 0
        AND t.service_id = '10'
    ),
    best_match AS (
        SELECT
            *,
            CASE
                WHEN ABS(predicted_unix - scheduled_prev) < ABS(predicted_unix - scheduled_same)
                 AND ABS(predicted_unix - scheduled_prev) < ABS(predicted_unix - scheduled_next)
                THEN scheduled_prev
                WHEN ABS(predicted_unix - scheduled_next) < ABS(predicted_unix - scheduled_same)
                THEN scheduled_next
                ELSE scheduled_same
            END AS scheduled_unix
        FROM delay_calc
    ),
    ranked AS (
        SELECT
            *,
            ROUND((predicted_unix - scheduled_unix) / 60.0, 1) AS delay_minutes,
            ROW_NUMBER() OVER (
                PARTITION BY trip_id, stop_sequence
                ORDER BY ABS(predicted_unix - scheduled_unix)
            ) AS snapshot_rank
        FROM best_match
    )
    SELECT
        trip_id,
        route_id,
        service_date,
        snapshot_ts,
        stop_sequence,
        stop_id,
        schedule_relationship,
        scheduled_arrival_time,
        scheduled_arrival_seconds,
        scheduled_unix,
        predicted_unix,
        predicted_ts,
        delay_minutes
    FROM ranked
    WHERE snapshot_rank = 1
""")

con.sql("SELECT COUNT(*) as rows FROM trip_delays").df()

,rows
0,42041


In [40]:
print(con.sql("""
    SELECT
        route_id,
        COUNT(*)                                                         AS total_stop_events,
        -- Strict: 0 to 3 min late
        ROUND(
            SUM(CASE WHEN delay_minutes BETWEEN 0 AND 3 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)
        , 1)                                                             AS otp_0_3,
        -- Standard: 1 min early to 5 min late
        ROUND(
            SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)
        , 1)                                                             AS otp_1_5,
        -- Lenient: 2 min early to 7 min late
        ROUND(
            SUM(CASE WHEN delay_minutes BETWEEN -2 AND 7 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)
        , 1)                                                             AS otp_2_7
    FROM trip_delays
    GROUP BY route_id
    ORDER BY route_id
""").df().to_string())

  route_id  total_stop_events  otp_0_3  otp_1_5  otp_2_7
0       23              21706     49.2     85.9     94.5
1       47              20335     44.2     79.8     91.3
